In [ ]:
#Intento 1: Funcionó. 
# Modelo base: Gradient Boosting con parámetros por defecto, probabilidades con "predict_proba", esperanza matemática de cada préstamosy selección por rentabilidad hasya ahotar el presupuesto. 

#Preparación de datos (la misma para todos los intentos): 
# Leakage: solo usamos meses del historial bancario anteriores a la fecha de solicitud (meses_atras >= 1).
# Ventana de 12 meses: las solicitudes de enero 2025 solo tienen 12 meses de historia previa. Usar 12 para todas hace que las features de train y test se calculen igual.
# Agregación por "ID_Solicitud" (no por usuario), porque un mismo usuario puede tener varios préstamos. 

import pandas as pd
import numpy as np

train = pd.read_csv('data/solicitudes_train.csv')
test  = pd.read_csv('data/solicitudes_test.csv')
banco = pd.read_csv('data/comportamiento_bancario.csv')

In [4]:
# Juntamos train y test para calcular las features igual en los dos
solicitudes = pd.concat([train, test])[['ID_Solicitud', 'ID_Usuario', 'Fecha_Solicitud']]

# Cada solicitud se cruza con todos los meses de su usuario
hist = solicitudes.merge(banco, on='ID_Usuario', how='left')

# Calculamos cuántos meses antes de la solicitud es cada registro de comportamiento bancario
fecha_sol = pd.to_datetime(hist['Fecha_Solicitud'])
fecha_mes = pd.to_datetime(hist['Mes_Referencia']) #mes de cada registro de comportamiento bancario
hist['meses_atras'] = (fecha_sol.dt.year - fecha_mes.dt.year) * 12 + (fecha_sol.dt.month - fecha_mes.dt.month)

# Usamos solo los 12 meses anteriores a la solicitud
VENTANA = 12
hist = hist[(hist['meses_atras'] >= 1) & (hist['meses_atras'] <= VENTANA)]

In [ ]:
hist['tuvo_atraso_30'] = hist['Dias_Atraso_Otras_Deudas'] > 30

feat = hist.groupby('ID_Solicitud').agg(
    saldo_prom      = ('Saldo_Promedio_Mensual', 'mean'),
    saldo_min       = ('Saldo_Promedio_Mensual', 'min'),
    saldo_std       = ('Saldo_Promedio_Mensual', 'std'),
    atraso_max      = ('Dias_Atraso_Otras_Deudas', 'max'),
    atraso_prom     = ('Dias_Atraso_Otras_Deudas', 'mean'),
    meses_atraso_30 = ('tuvo_atraso_30', 'sum'),   # cuántos meses debió más de 30 días
    meses_informados= ('Mes_Referencia', 'count'), # cuántos meses de registro tenés
)

# Filtramos por los ultimos 3 meses para evaluar el comportamiento reciente
ult3 = hist[hist['meses_atras'] <= 3]
feat_ult3 = ult3.groupby('ID_Solicitud').agg(
    saldo_prom_ult3 = ('Saldo_Promedio_Mensual', 'mean'),
    atraso_max_ult3 = ('Dias_Atraso_Otras_Deudas', 'max'),
)

# Unimos las features de los ultimos 3 meses con las features generales
feat = feat.join(feat_ult3)
feat['tendencia_saldo'] = feat['saldo_prom_ult3'] / feat['saldo_prom']  # >1 = viene mejorando
feat = feat.reset_index()

In [6]:
# Unimos las features con los datasets de train y test
train_f = train.merge(feat, on='ID_Solicitud', how='left')
test_f  = test.merge(feat,  on='ID_Solicitud', how='left')

print(train_f.shape, test_f.shape)
print(train_f.isna().sum()[lambda s: s > 0])

train_f.groupby('Estado_Final')[['atraso_max', 'meses_atraso_30', 'saldo_prom']].mean()

(32416, 18) (3061, 17)
saldo_prom_ult3    4
atraso_max_ult3    4
tendencia_saldo    4
dtype: int64


,atraso_max,meses_atraso_30,saldo_prom
Estado_Final,,,
0,45.567202,1.511700,2731.242914
1,31.285594,0.769059,5892.787649


In [7]:
# Verificamos si a partir de sus sueldo, puede pagar la cuota del préstamo y si su saldo promedio es mayor que su sueldo
for df in (train_f, test_f):
    df['cuota_sobre_ingreso'] = df['Monto_Solicitado'] / df['Ingreso_Mensual']   
    df['saldo_sobre_ingreso'] = df['saldo_prom'] / df['Ingreso_Mensual']

# Motivo_Prestamo es texto: lo pasamos a categoría, con las mismas categorías en train y test
motivos = sorted(train_f['Motivo_Prestamo'].unique())
for df in (train_f, test_f):
    df['Motivo_Prestamo'] = pd.Categorical(df['Motivo_Prestamo'], categories=motivos)

FEATURES = ['Monto_Solicitado', 'Tasa_Interes', 'Ingreso_Mensual', 'Motivo_Prestamo',
            'saldo_prom', 'saldo_min', 'saldo_std', 'atraso_max', 'atraso_prom', 'meses_atraso_30',
            'meses_informados', 'saldo_prom_ult3', 'atraso_max_ult3', 'tendencia_saldo',
            'cuota_sobre_ingreso', 'saldo_sobre_ingreso']

# Datasets de train y test para el modelo 
X      = train_f[FEATURES]
y      = train_f['Estado_Final']
grupos = train_f['ID_Usuario']
X_test = test_f[FEATURES]

PRESUPUESTO = 2_000_000

In [ ]:
# Entrenamiento del modelo de Gradient Boosting regularizado
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# Dividimos 80% para entrenar, 20% para validar
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Entrenamos un modelo de Gradient Boosting con HistGradientBoostingClassifier
modelo = HistGradientBoostingClassifier(categorical_features='from_dtype', random_state=42)
modelo.fit(X_train, y_train)

# Probabilidad de que pague (columna 1 de predict_proba)
p_val = modelo.predict_proba(X_val)[:, 1]

# AUC mide que tan bien el modelo ordena a los clientes
print('AUC validación:', round(roc_auc_score(y_val, p_val), 4))

AUC validación: 0.838


In [ ]:
# Regla de decisión sobre validación
# Para cada préstamo: E = p · Monto · Tasa − (1 − p) · Monto. Aprobamos si E > 0, ordenando por rentabilidad (E / Monto) hasta llegar a USD 2.000.000.

# Creamos un dataset con la información de validación y la probabilidad de pago
val = train_f.loc[X_val.index].copy()
val['p_pago'] = p_val

# Esperanza matemática de cada préstamo:
# con prob. p gano monto*tasa; con prob. (1-p) pierdo el monto
val['ganancia_esperada'] = (val['p_pago'] * val['Monto_Solicitado'] * val['Tasa_Interes']
                            - (1 - val['p_pago']) * val['Monto_Solicitado'])

# Ganancia esperada por cada dólar prestado
val['rentabilidad'] = val['ganancia_esperada'] / val['Monto_Solicitado']

# Ordenamos de más a menos rentable y vamos sumando montos
val = val.sort_values('rentabilidad', ascending=False)
val['acumulado'] = val['Monto_Solicitado'].cumsum()

# Aprobamos si conviene (esperanza > 0) y si todavía entra en el presupuesto
val['aprobado'] = ((val['ganancia_esperada'] > 0) & (val['acumulado'] <= PRESUPUESTO)).astype(int)

# Vemos qué tan bien le fue al modelo comparando con lo que pasó en realidad (Estado_Final)
val['resultado'] = np.where(val['Estado_Final'] == 1,
                            val['Monto_Solicitado'] * val['Tasa_Interes'],
                            -val['Monto_Solicitado'])

# Calculamos cuántos préstamos aprobados hubo, cuánto se prestó y cuánto se ganó realmente
aprobados = val[val['aprobado'] == 1]
print('Préstamos aprobados:', len(aprobados))
print('Monto prestado:', aprobados['Monto_Solicitado'].sum())
print('Ganancia real:', round(aprobados['resultado'].sum()))

Préstamos aprobados: 73
Monto prestado: 1987000
Ganancia real: 478581


In [9]:
modelo_final = HistGradientBoostingClassifier(categorical_features='from_dtype', random_state=42)
modelo_final.fit(X, y)

,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.1
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",100
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""Categorical"" and ""Enum"" are considered to be categorical features. The input must be a dataframe that is supported by narwhals (or supports it): :func:`narwhals.from_native` must work. This is the case, for instance, for pandas and polars DataFrames.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing val

In [10]:
# Creamos un dataset con la información de test y la probabilidad de pago
sub = test_f[['ID_Solicitud', 'Monto_Solicitado', 'Tasa_Interes']].copy()
sub['p_pago'] = modelo_final.predict_proba(X_test)[:, 1]

sub['ganancia_esperada'] = (sub['p_pago'] * sub['Monto_Solicitado'] * sub['Tasa_Interes']
                            - (1 - sub['p_pago']) * sub['Monto_Solicitado'])
sub['rentabilidad'] = sub['ganancia_esperada'] / sub['Monto_Solicitado']

# Ordenamos por rentabilidad y vamos sumando montos hasta llegar al presupuesto
sub = sub.sort_values('rentabilidad', ascending=False)
sub['acumulado'] = sub['Monto_Solicitado'].cumsum()
sub['aprobado'] = ((sub['ganancia_esperada'] > 0) & (sub['acumulado'] <= PRESUPUESTO)).astype(int)

# Controlamos no pasarnos del presupuesto
monto_total = sub.loc[sub['aprobado'] == 1, 'Monto_Solicitado'].sum()
print('Aprobados:', sub['aprobado'].sum(), 'de', len(sub))
print(f'Monto total aprobado: $ {monto_total:,.0f}')
assert monto_total <= PRESUPUESTO

sub = sub.sort_values('ID_Solicitud')
sub = sub.rename(columns={'aprobado': 'Decision_Aprobacion'})
sub[['ID_Solicitud', 'Decision_Aprobacion']].to_csv('submissions/entrega_01.csv', index=False)

Aprobados: 653 de 3061
Monto total aprobado: $ 1,999,600
